In [ ]:
import sys , os
import pandas as pd
import re


sys.path.append('..')

import time
from typing import Literal, Optional
from pydantic import BaseModel
from utils.prompts import render
from utils.llm_client import LLMClient
from utils.token_utils import count_messages_tokens
from utils.logging_utils import log_llm_call
from utils.router import pick_model, should_use_reasoning_model
from IPython.display import Markdown, display
from utils.json_utils import format_pydantic_schema_for_prompt, parse_json_with_pydantic

# Part 01

In [ ]:
file_path = '../data/Sample Messages.txt'
with open(file_path, 'r', encoding='utf-8') as f:
    raw_text = f.read()


cleaned_text = re.sub(r'\\s*', '', raw_text)
messages_list = [line.strip() for line in cleaned_text.split('\n') if line.strip()]

print(f"Total messages to process: {len(messages_list)}\n")


model = pick_model('google', 'general')
client = LLMClient('google', model)


results = []

examples = """
Example 1:
Message: "Breaking News: Kelani River level at 9m."
Output: District: Colombo | Intent: Info | Priority: Low

Example 2:
Message: "We are trapped on the roof with 3 kids!"
Output: District: None | Intent: Rescue | Priority: High

Example 3:
Message: "Does anyone have extra dry rations for the camp in Gampaha?"
Output: District: Gampaha | Intent: Supply | Priority: High

Example 4:
Message: "Just arrived in Galle. Weather is nice."
Output: District: Galle | Intent: Other | Priority: Low
"""


for msg in messages_list[:4]:
    

    prompt_text, spec = render(
        'few_shot.v1',
        role='rescue, supply, info, other classifier',
        examples=examples,
        query=f'Message: "{msg}"',
        constraints='Follow the pattern in examples: provide output in the format "District: <district> | Intent: <intent> | Priority: <priority>". If the district is not mentioned, use "None".',
        format='District: {{district}} | Intent: {{intent}} | Priority: {{priority}}'
    )

    chat_messages = [{'role': 'user', 'content': prompt_text}]
    

    response = client.chat(chat_messages, temperature=0.1) 
    

    output_text = response['text'].strip()
    

    results.append({
        'Original Message': msg,
        'Classification': output_text
    })
    
    print(f"Processed: {msg[:40]}... -> {output_text}")
    

    if 'latency_ms' in response:
        log_llm_call('gemini', model, 'few_shot', response['latency_ms'], response.get('usage', {}))


os.makedirs('../output', exist_ok=True)
df = pd.DataFrame(results)


output_file = '../output/classified_messages.xlsx'
df.to_excel(output_file, index=False)

print(f"\nDone! All results saved to {output_file}")

# Part 02

In [ ]:
print("--- Starting Part 2: Stability Experiment ---\n")


file_path = '../data/Scenarios.txt'
with open(file_path, 'r', encoding='utf-8') as f:
    raw_scenarios = f.read()


cleaned_scenarios = re.sub(r'\\s*', '', raw_scenarios)


scenarios_list = [s.strip() for s in cleaned_scenarios.split('SCENARIO') if s.strip()]
scenarios_list = ['SCENARIO ' + s for s in scenarios_list]

print(f"Found {len(scenarios_list)} scenarios to test.\n")


reasoning_model = pick_model('google', 'cot') 
print(f'Using reasoning model: {reasoning_model}\n')
client_reasoning = LLMClient('google', reasoning_model)


for scenario in scenarios_list:
    print("="*80)
    print(f"Testing Scenario:\n{scenario[:150]}...\n") 
    print("="*80)
    

    prompt_text, spec = render(
        'cot_reasoning.v1', 
        role='expert emergency response coordinator',
        problem=scenario
    )
    
    chat_messages = [{'role': 'user', 'content': prompt_text}]
    
    # ---------------------------------------------------------
    # SAFE MODE (Temperature 0.0) 
    # ---------------------------------------------------------
    print("\n [ SAFE MODE | Temperature = 0.0 ]\n")
    response_safe = client_reasoning.chat(
        chat_messages, 
        temperature=0.0, 
        max_tokens=spec.max_tokens
    )
    display(Markdown(response_safe['text']))
    
    # ---------------------------------------------------------
    # CHAOS MODE (Temperature 1.0) - Running 3 times
    # ---------------------------------------------------------
    print("\n\n [ CHAOS MODE | Temperature = 1.0 ] (Running 3 times)")
    print("-" * 60)
    
    for i in range(1, 4):
        print(f"\n--- Chaos Run {i} ---")
        response_chaos = client_reasoning.chat(
            chat_messages, 
            temperature=1.0, 
            max_tokens=spec.max_tokens
        )
        display(Markdown(response_chaos['text']))
        
    print("\n" + "#"*80 + "\n")

# Part 03

In [ ]:
from IPython.display import Markdown, display
print("--- Starting Part 3: The Logistics Commander (CoT & ToT) ---\n")


file_path = '../data/Incidents.txt'
with open(file_path, 'r', encoding='utf-8') as f:
    raw_incidents = f.read()


cleaned_incidents = re.sub(r'\\s*', '', raw_incidents).strip()
print("Data Loaded Successfully!\n")
print("-" * 60)

# ==========================================
# STEP A: CoT - Scoring 
# ==========================================
print("Executing Step A (CoT - Scoring)...")

reasoning_model = pick_model('google', 'cot')
client_cot = LLMClient('google', reasoning_model)


step_a_logic = f"""
Analyze the following 3 incidents row by row and assign a Priority Score (1-10) to each.
Logic:
- Base Score: 5
- +2 if Age > 60 or Age < 5
- +3 if Need == "Rescue" (Life Threat)
- +1 if Need == "Insulin/Medicine"

Incidents Data:
{cleaned_incidents}
"""

prompt_a, spec_a = render(
    'cot_reasoning.v1', 
    role='expert emergency response coordinator',
    problem=step_a_logic
)

response_a = client_cot.chat([{'role': 'user', 'content': prompt_a}], temperature=0.0, max_tokens=spec_a.max_tokens)


display(Markdown("### Step A: CoT - Scoring Results"))
display(Markdown(response_a['text']))
print("-" * 60)

# ==========================================
# STEP B: ToT - Strategy 
# ==========================================
print("Executing Step B (ToT - Strategy)...")


strategy_model = pick_model('google', 'tot')
client_tot = LLMClient('google', strategy_model)

step_b_logic = f"""
Based on the scores calculated in Step A, determine the optimal rescue route.
Situation: You have ONE rescue boat starting at Ragama.

Original Incidents Data (Locations):
{cleaned_incidents}

Scores calculated in Step A:
{response_a['text']}

Constraints - Travel times:
- Ragama -> Ja-Ela: 10 mins
- Ja-Ela -> Gampaha: 40 mins

Task: Explore 3 distinct branches:
- Branch 1: Save the highest score first (Greedy).
- Branch 2: Save closest first (Speed).
- Branch 3: Save furthest first (Logistics).

Goal: Maximize total priority score saved within the shortest time. Assume the boat can handle one incident per stop.
Select the optimal route and explain why.

CRITICAL INSTRUCTION: Keep your reasoning STRICTLY CONCISE. Do not write long paragraphs. Summarize each branch in a maximum of 2 to 3 bullet points. Just output the math and the final decision.
"""

prompt_b, spec_b = render(
    'tot_reasoning.v1', 
    role='expert logistics commander',
    problem=step_b_logic,
    branches='3'
)

response_b = client_tot.chat([{'role': 'user', 'content': prompt_b}], temperature=0.0, max_tokens=spec_b.max_tokens)


display(Markdown("### Step B: ToT - Strategy Results"))
display(Markdown(response_b['text']))

print("\n--- Part 3 Execution Done! ---")

# part 04

In [ ]:
print("--- Starting Part 4: The 'Budget Keeper' (Token Economics) ---\n")


long_spam_message = """
URGENT WARNING!!! Please forward this message to 20 people in the next 5 minutes! 
If you ignore this, your computer will crash and you will lose all your data! 
This is not a joke! Did you know that in 1999, a person ignored this and their phone exploded? 
Also, don't forget to click the link below to claim your free $1,000,000 lottery prize. 
Save the turtles, buy crypto, subscribe to my channel, like and share!
""" * 5  


messages = [{'role': 'user', 'content': long_spam_message}]
provider = 'google'
general_model = pick_model(provider, 'general')


token_counts = count_messages_tokens(messages, provider, general_model)
total_tokens = token_counts['estimated_total']

print(f"Incoming Message Token Count: {total_tokens}")


TOKEN_LIMIT = 150

if total_tokens > TOKEN_LIMIT:
  
    print("\n BLOCKED/TRUNCATED: Message exceeds 150 tokens!")
    print("Initiating overflow summarization...\n")
    

    client_budget = LLMClient(provider, general_model)
    
    prompt_text, spec = render(
        'overflow_summarize.v1', 
        max_tokens_context=50,                  
        context=long_spam_message,              
        task="Extract the main intention of the sender.", 
        format="Provide ONLY the final extracted intention in a single concise sentence. Do NOT print 'Step 1', 'Step 2' or the intermediate summary."
    )
    
    chat_msgs = [{'role': 'user', 'content': prompt_text}]
    response = client_budget.chat(chat_msgs, temperature=spec.temperature, max_tokens=spec.max_tokens)
    
    display(Markdown("### Summarized Safe Output:"))
    display(Markdown(response['text']))

else:
    print("\nMessage is within limits. Processing normally...")

print("\n--- Part 4 Execution Done! ---")



# part 05

In [ ]:
print("--- Starting Part 5: The 'News Feed' Extraction Pipeline ---\n")

# ==========================================
# 1. Define Schema (Pydantic Model)
# ==========================================
class CrisisEvent(BaseModel):

    district: Literal['Colombo', 'Gampaha', 'Kandy', 'Kalutara', 'Galle', 'Matara', 'Ratnapura', 'Kegalle', 'Anuradhapura']
    flood_level_meters: Optional[float] = None
    victim_count: int = 0
    main_need: str
    status: Literal['Critical', 'Warning', 'Stable']

# ==========================================
# 2. Process Feed
# ==========================================
input_file = '../data/News Feed.txt' 
output_dir = '../output'
output_file = f'{output_dir}/flood_report.xlsx'


os.makedirs(output_dir, exist_ok=True)

provider = 'google'
ext_model = pick_model(provider, 'extraction')
client_ext = LLMClient(provider, ext_model)

valid_events = []
schema_str = format_pydantic_schema_for_prompt(CrisisEvent)


try:
    with open(input_file, 'r', encoding='utf-8') as f:
     
        lines = [line.strip() for line in f if line.strip()]
except FileNotFoundError:
    print(f" Error: Could not find {input_file}. Please ensure the file exists.")
    lines = []

print(f"Found {len(lines)} news items. Starting extraction...\n")


for i, line in enumerate(lines):
    print(f"Processing Item {i+1}...")
    
    # Use json_extract.v1 to extract structured data from the news feed line
    prompt_text, spec = render(
        'json_extract.v1',
        schema=schema_str,
        text=line
    )
    
    
    response = client_ext.chat(
        [{'role': 'user', 'content': prompt_text}],
        temperature=spec.temperature,
        max_tokens=spec.max_tokens
    )
    
    # using Pydantic to validate the extracted JSON data
    success, model_instance, error = parse_json_with_pydantic(response['text'], CrisisEvent)
    
    if success:
        valid_events.append(model_instance.model_dump())
        print(f"   Valid: {model_instance.district} | Status: {model_instance.status} | Need: {model_instance.main_need}")
    else:
    
        print(f"   Invalid: Skipped. Reason: {error}")

    time.sleep(30)  

print("-" * 60)

# ==========================================
# 3. Save to Excel
# ==========================================
if valid_events:
  
    df = pd.DataFrame(valid_events)
    
    
    df.to_excel(output_file, index=False)
    
    print(f"\n Extraction Complete! Successfully saved {len(valid_events)} records to '{output_file}'.")
    display(df.head()) 
else:
    print("\n No valid events were extracted to save.")

print("\n--- Part 5 Execution Done! ---")